<a href="https://colab.research.google.com/github/issacridhin/LabWorks/blob/LLM/2348546_LLM_MiniProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this project, we're using the **T5 (Text-To-Text Transfer Transformer)** model to perform language translation from English to French. The T5 model is versatile, capable of handling a variety of NLP tasks by converting them into a text-to-text format.

The **T5 (Text-To-Text Transfer Transformer)** model, developed by Google Research, is a versatile transformer-based model designed to handle various natural language processing tasks. It treats every problem as a text-to-text problem, meaning it converts all inputs into text and produces outputs as text, whether it's translation, summarization, or question answering. This unified approach simplifies the model's architecture and training, allowing it to excel across a range of tasks by leveraging its extensive pretraining on diverse text corpora. T5’s flexibility and effectiveness make it a powerful tool for numerous NLP applications.

In [ ]:
# Install the required libraries
!pip install transformers datasets sentencepiece sacrebleu

We are using a subset of the WMT14 dataset, specifically the French-English translation pair. This dataset is widely used in the field of machine translation and contains high-quality parallel sentences in both languages.

In [ ]:
from datasets import load_dataset

# Load the dataset from Hugging Face Datasets
dataset = load_dataset("wmt14", "fr-en", split="train[:1%]")

# Display the first few examples
dataset = dataset.shuffle(seed=42).select([i for i in range(1000)])  # Sampling 1000 sentences
print(dataset[0])

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Error while downloading from https://huggingface.co/datasets/wmt14/resolve/b199e406369ec1b7634206d3ded5ba45de2fe696/fr-en/train-00015-of-00030.parquet: HTTPSConnectionPool(host='cdn-lfs.huggingface.co', port=443): Read timed out.
Trying to resume download...
Trying to resume download...


Generating train split:   0%|          | 0/40836715 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

{'translation': {'en': 'We must not bow down before the agricultural or chemical industry lobbyists; on the contrary, we must use our vote to promote and express that fact that we want the polluter pays principle and, hence, cost-covering prices.', 'fr': "Nous ne pouvons pas nous mettre à genoux devant les intérêts des lobbies de l'industrie agricole et chimique ; nous devons plutôt proclamer et exprimer par notre vote que nous voulons le principe du pollueur payeur et, ce faisant, des prix qui couvrent les coûts."}}


#Data Preprocessing

To prepare the text for translation, we:

Clean the text: We remove unnecessary whitespace and non-alphanumeric

*   Clean the text: We remove unnecessary whitespace and non-alphanumeric characters to ensure that the input text is clean and well-formatted.
*   Tokenization: We tokenize the sentences using the T5Tokenizer, which converts text into a format that the T5 model can understand.




In [ ]:
import re
import nltk
from transformers import T5Tokenizer

# Download and initialize the NLTK tokenizer
nltk.download('punkt')
from nltk.tokenize import sent_tokenize, word_tokenize

# Initialize the tokenizer
tokenizer = T5Tokenizer.from_pretrained('t5-small')

def clean_text(text):
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove non-alphanumeric characters (excluding punctuation)
    text = re.sub(r"[^\w\s\.,!?\'\"-]", '', text)

    # Lowercase the text
    text = text.lower()

    # Tokenize sentences
    sentences = sent_tokenize(text)

    # Tokenize words and remove short tokens (e.g., single characters)
    sentences = [word_tokenize(sentence) for sentence in sentences]
    sentences = [' '.join([word for word in sentence if len(word) > 1]) for sentence in sentences]

    # Recombine sentences
    text = ' '.join(sentences)

    return text

def preprocess_function(examples):
    # Extracting English and French sentences
    english_texts = [clean_text(ex['en']) for ex in examples["translation"]]
    french_texts = [clean_text(ex['fr']) for ex in examples["translation"]]

    # Preparing inputs and targets
    inputs = ["translate English to French: " + text for text in english_texts]
    targets = french_texts

    # Apply tokenization with padding
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding='max_length', return_tensors='pt')
    labels = tokenizer(targets, max_length=128, truncation=True, padding='max_length', return_tensors='pt')

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


We are using the T5-small model, which is a smaller version of the T5 model, making it faster and more efficient for our task. We fine-tune this model on our dataset by training it for 3 epochs.

The model is trained using the Trainer class from the transformers library. We define specific training arguments, such as the learning rate, batch size, and the number of epochs, to optimize the model's performance.

We can also use T5-Base to improve the model performance but it would take lot of time to run.

In [ ]:
from transformers import T5ForConditionalGeneration, Trainer, TrainingArguments

# Load the T5 model
model = T5ForConditionalGeneration.from_pretrained('t5-small')

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Fine-tune the model
trainer.train()


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss
500,2.183000


TrainOutput(global_step=750, training_loss=1.7445065511067708, metrics={'train_runtime': 88.2947, 'train_samples_per_second': 33.977, 'train_steps_per_second': 8.494, 'total_flos': 99920314368000.0, 'train_loss': 1.7445065511067708, 'epoch': 3.0})

Once trained, the model can generate translations for new English sentences. We simply provide an English sentence, and the model outputs the corresponding French translation.

In [ ]:
import torch

# Ensure the model is on the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def generate_translation(text):
    input_ids = tokenizer.encode("translate English to French: " + text, return_tensors="pt").to(device)
    with torch.no_grad():  # Disable gradient calculation
        outputs = model.generate(input_ids)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test the model on new English sentences
sample_sentences = ["My name is Ridhin.", "I am learning how to translate text.", "Machine translation is very interesting."]
for sentence in sample_sentences:
    print(f"English: {sentence}")
    print(f"French: {generate_translation(sentence)}\n")


English: My name is Ridhin.


/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1249: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


French: Mon nom est Ridhin.

English: I am learning how to translate text.
French: Je m'apprend à traduire le texte.

English: Machine translation is very interesting.
French: La traduction des machines est très intéressante.

